%matplotlib inline
from pathlib import Path

import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from src.classifiers import benchmark_classifiers, extract_tree_feature_importance
from src.data_loader import load_telco_data, split_train_val_test
from src.evaluation import classification_metrics, plot_confusion_matrix, plot_feature_importance, plot_roc_curve
from src.preprocessing import handle_class_imbalance, prepare_train_val_test_features
from src.utils import PROJECT_ROOT, ensure_directory, set_random_seed

set_random_seed(42)
figures_dir = ensure_directory(PROJECT_ROOT / 'reports' / 'figures')
models_dir = ensure_directory(PROJECT_ROOT / 'models')

frame = load_telco_data()
train_frame, validation_frame, test_frame = split_train_val_test(frame, target_column='Churn', stratify=True, random_state=42)
X_train, y_train, X_validation, y_validation, X_test, y_test, preprocessor, feature_names = prepare_train_val_test_features(
    train_frame,
    validation_frame,
    test_frame,
    target_column='Churn',
    drop_columns=['customerID'],
    scaler='none',
)

X_smote, y_smote = handle_class_imbalance(X_train, y_train, method='smote', random_state=42)
X_under, y_under = handle_class_imbalance(X_train, y_train, method='undersample', random_state=42)

baseline_model = Pipeline(steps=[('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=3000, class_weight='balanced', random_state=42))])
for label, features, targets in [('SMOTE', X_smote, y_smote), ('Undersampled', X_under, y_under)]:
    baseline_model.fit(features, targets)
    validation_predictions = baseline_model.predict(X_validation)
    validation_probabilities = baseline_model.predict_proba(X_validation)[:, 1]
    print(label, classification_metrics(y_validation, validation_predictions, validation_probabilities))

comparison_frame, best_classifier, best_row = benchmark_classifiers(
    X_train,
    y_train,
    X_validation,
    y_validation,
    X_test,
    y_test,
    random_state=42,
    save_path=models_dir / 'best_classifier.pkl',
)

print(comparison_frame[['model', 'validation_f1', 'validation_roc_auc', 'test_f1', 'test_roc_auc', 'training_time']])
print('Best model on validation set:', best_row['model'])

test_predictions = best_classifier.predict(X_test)
test_probabilities = best_classifier.predict_proba(X_test)[:, 1]
print('Test metrics:', classification_metrics(y_test, test_predictions, test_probabilities))

plot_confusion_matrix(y_test, test_predictions, 'Best Classification Model: Confusion Matrix', figures_dir / 'classification_confusion_matrix.png')
plot_roc_curve(y_test, test_probabilities, 'Best Classification Model: ROC Curve', figures_dir / 'classification_roc_curve.png')

try:
    feature_importance = extract_tree_feature_importance(best_classifier, feature_names)
    plot_feature_importance(feature_importance.index, feature_importance.values, 'Best Classification Model: Feature Importance', figures_dir / 'classification_feature_importance.png')
except ValueError:
    print('Best classifier is not tree-based, so feature importance is skipped.')

top_targets = X_test.assign(actual=y_test, probability=test_probabilities).sort_values('probability', ascending=False).head(100)
print('Top 100 high-risk customers sample:')
print(top_targets[['actual', 'probability']].head(10))

In [ ]:
%matplotlib inline
from pathlib import Path

import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

from src.classifiers import benchmark_classifiers, extract_tree_feature_importance
from src.data_loader import load_telco_data, split_train_val_test
from src.evaluation import classification_metrics, plot_confusion_matrix, plot_feature_importance, plot_roc_curve
from src.preprocessing import handle_class_imbalance, prepare_train_val_test_features
from src.utils import PROJECT_ROOT, ensure_directory, set_random_seed

set_random_seed(42)
figures_dir = ensure_directory(PROJECT_ROOT / 'reports' / 'figures')
models_dir = ensure_directory(PROJECT_ROOT / 'models')

frame = load_telco_data()
train_frame, validation_frame, test_frame = split_train_val_test(frame, target_column='Churn', stratify=True, random_state=42)
X_train, y_train, X_validation, y_validation, X_test, y_test, preprocessor, feature_names = prepare_train_val_test_features(
    train_frame,
    validation_frame,
    test_frame,
    target_column='Churn',
    drop_columns=['customerID'],
    scaler='none',
)

X_smote, y_smote = handle_class_imbalance(X_train, y_train, method='smote', random_state=42)
X_under, y_under = handle_class_imbalance(X_train, y_train, method='undersample', random_state=42)

baseline_model = LogisticRegression(max_iter=3000, class_weight='balanced', random_state=42)
for label, features, targets in [('SMOTE', X_smote, y_smote), ('Undersampled', X_under, y_under)]:
    baseline_model.fit(features, targets)
    validation_predictions = baseline_model.predict(X_validation)
    validation_probabilities = baseline_model.predict_proba(X_validation)[:, 1]
    print(label, classification_metrics(y_validation, validation_predictions, validation_probabilities))

comparison_frame, best_classifier, best_row = benchmark_classifiers(
    X_train,
    y_train,
    X_validation,
    y_validation,
    X_test,
    y_test,
    random_state=42,
    save_path=models_dir / 'best_classifier.pkl',
)

print(comparison_frame[['model', 'validation_f1', 'validation_roc_auc', 'test_f1', 'test_roc_auc', 'training_time']])
print('Best model on validation set:', best_row['model'])

test_predictions = best_classifier.predict(X_test)
test_probabilities = best_classifier.predict_proba(X_test)[:, 1]
print('Test metrics:', classification_metrics(y_test, test_predictions, test_probabilities))

plot_confusion_matrix(y_test, test_predictions, 'Best Classification Model: Confusion Matrix', figures_dir / 'classification_confusion_matrix.png')
plot_roc_curve(y_test, test_probabilities, 'Best Classification Model: ROC Curve', figures_dir / 'classification_roc_curve.png')

try:
    feature_importance = extract_tree_feature_importance(best_classifier, feature_names)
    plot_feature_importance(feature_importance.index, feature_importance.values, 'Best Classification Model: Feature Importance', figures_dir / 'classification_feature_importance.png')
except ValueError:
    print('Best classifier is not tree-based, so feature importance is skipped.')

top_targets = X_test.assign(actual=y_test, probability=test_probabilities).sort_values('probability', ascending=False).head(100)
print('Top 100 high-risk customers sample:')
print(top_targets[['actual', 'probability']].head(10))